# AffectLab final full-data multimodal models

Train one context-text model and one audio model on all 5,531 unique benchmark utterances. Epoch counts are fixed from the median completed cross-validation epochs. These artifacts are for inference packaging; training metrics are not evaluation results.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime; A100 or L4 is recommended.'
print(torch.cuda.get_device_name(0))

In [ ]:
import base64, hashlib, json, os, statistics, subprocess, sys
from pathlib import Path
from google.colab import auth, userdata
PROJECT_ID, BUCKET = 'cat-behaviour-research', 'affectlab-research-raluca-biras'
TEXT_EXPERIMENT = 'iemocap_benchmark4_context3_deberta_v3_small'
AUDIO_EXPERIMENT = 'iemocap_benchmark4_audio_wav2vec2_base'
auth.authenticate_user()
subprocess.run(['gcloud','config','set','project',PROJECT_ID],check=True)
repo=Path('/content/emotion-aware-role-play-model')
token=userdata.get('GITHUB_TOKEN')
if not token: raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets.')
header=base64.b64encode(f'x-access-token:{token}'.encode()).decode(); option=f'http.extraHeader=Authorization: Basic {header}'
url='https://github.com/ralucabiras/emotion-aware-role-play-model.git'
if not repo.exists(): subprocess.run(['git','-c',option,'clone',url,str(repo)],check=True)
else: subprocess.run(['git','-C',str(repo),'-c',option,'pull','--ff-only'],check=True)
del token,header,option; os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-ml.txt'],check=True)
from huggingface_hub import login
hf_token=userdata.get('HF_TOKEN')
if hf_token: login(token=hf_token,add_to_git_credential=False)

In [ ]:
epoch_root=Path('/content/cv-metrics')
completed={'text':[],'audio':[]}
for fold in range(1,6):
    for family,experiment,modality in (('iemocap-text',TEXT_EXPERIMENT,'text'),('iemocap-audio',AUDIO_EXPERIMENT,'audio')):
        path=epoch_root/modality/f'fold-{fold}.json'; path.parent.mkdir(parents=True,exist_ok=True)
        subprocess.run(['gcloud','storage','cp',f'gs://{BUCKET}/runs/{family}/{experiment}/fold-{fold}/metrics.json',str(path)],check=True)
        completed[modality].append(float(json.loads(path.read_text())['train_metrics']['epoch']))
TEXT_EPOCHS=max(1,round(statistics.median(completed['text'])))
AUDIO_EPOCHS=max(1,round(statistics.median(completed['audio'])))
print({'completed_cv_epochs':completed,'final_text_epochs':TEXT_EPOCHS,'final_audio_epochs':AUDIO_EPOCHS})

In [ ]:
TEXT_DATA=Path('/content/iemocap-context-data'); AUDIO_DATA=Path('/content/iemocap-audio-data'); AUDIO_ROOT=Path('/content/iemocap-audio'); OUTPUT=Path('/content/final-models')
subprocess.run(['gcloud','storage','rsync','--recursive',f'gs://{BUCKET}/data/processed/iemocap-text-v2-context3',str(TEXT_DATA)],check=True)
subprocess.run(['gcloud','storage','rsync','--recursive',f'gs://{BUCKET}/data/processed/iemocap-text-v1',str(AUDIO_DATA)],check=True)
audio_gcs=f'gs://{BUCKET}/data/processed/iemocap-audio-benchmark4-v1'; manifest_path=Path('/content/audio-manifest.json'); bundle=Path('/content/iemocap-audio.tar.gz')
subprocess.run(['gcloud','storage','cp',f'{audio_gcs}/iemocap-benchmark4-audio-v1.tar.gz.manifest.json',str(manifest_path)],check=True)
subprocess.run(['gcloud','storage','cp',f'{audio_gcs}/iemocap-benchmark4-audio-v1.tar.gz',str(bundle)],check=True)
manifest=json.loads(manifest_path.read_text()); digest=hashlib.sha256()
with bundle.open('rb') as handle:
    while chunk:=handle.read(16*1024*1024): digest.update(chunk)
assert digest.hexdigest().upper()==manifest['bundle_sha256']
from ml.preprocessing.iemocap_audio import extract_bundle
extract_bundle(bundle,AUDIO_ROOT,manifest['files'])

In [ ]:
config=repo/'configs/iemocap_final_multimodal.json'
subprocess.run([sys.executable,'-m','ml.training.train_iemocap_final','--config',str(config),'--modality','text','--data-root',str(TEXT_DATA),'--output-root',str(OUTPUT),'--epochs',str(TEXT_EPOCHS)],check=True)
text_dir=OUTPUT/'iemocap_benchmark4_final_multimodal'/'text'
subprocess.run(['gcloud','storage','rsync','--recursive',str(text_dir),f'gs://{BUCKET}/models/iemocap-benchmark4-final-v1/text'],check=True)
print(json.loads((text_dir/'training_manifest.json').read_text()))

In [ ]:
subprocess.run([sys.executable,'-m','ml.training.train_iemocap_final','--config',str(config),'--modality','audio','--data-root',str(AUDIO_DATA),'--audio-root',str(AUDIO_ROOT),'--output-root',str(OUTPUT),'--epochs',str(AUDIO_EPOCHS)],check=True)
audio_dir=OUTPUT/'iemocap_benchmark4_final_multimodal'/'audio'
subprocess.run(['gcloud','storage','rsync','--recursive',str(audio_dir),f'gs://{BUCKET}/models/iemocap-benchmark4-final-v1/audio'],check=True)
print(json.loads((audio_dir/'training_manifest.json').read_text()))

In [ ]:
smoke_output=OUTPUT/'iemocap_benchmark4_final_multimodal'/'smoke_test.json'
subprocess.run([sys.executable,'-m','ml.training.smoke_iemocap_final','--config',str(config),'--text-model',str(text_dir/'model'),'--audio-model',str(audio_dir/'model'),'--data-root',str(TEXT_DATA),'--audio-root',str(AUDIO_ROOT),'--examples-per-class','32','--output',str(smoke_output)],check=True)
smoke=json.loads(smoke_output.read_text()); display(smoke)
subprocess.run(['gcloud','storage','cp',str(smoke_output),f'gs://{BUCKET}/models/iemocap-benchmark4-final-v1/smoke_test.json'],check=True)

## Completion check

Confirm both private model directories and `smoke_test.json` were uploaded. The smoke sample is an artifact-integrity diagnostic on training rows, not an evaluation result. The cross-validation and equal-fusion results remain the dissertation evaluation.